In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from collections import Counter

In [2]:
def make_long_tailed(dataset, imbalance_ratio=100):
    targets = np.array(dataset.targets)
    classes = np.unique(targets)

    max_num = max([np.sum(targets == c) for c in classes])

    img_num_per_cls = []
    for i, cls in enumerate(classes):
        num = max_num * (imbalance_ratio ** (-i / (len(classes)-1)))
        img_num_per_cls.append(int(num))

    new_data, new_targets = [], []

    for cls, num in zip(classes, img_num_per_cls):
        idx = np.where(targets == cls)[0]
        np.random.shuffle(idx)
        selected = idx[:num]

        new_data.append(dataset.data[selected])
        new_targets.extend([cls]*num)

    dataset.data = np.concatenate(new_data)
    dataset.targets = new_targets
    return dataset

In [3]:
class SimCLR(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.encoder = backbone
        self.projector = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return z

In [4]:
def get_backbone():
    model = torchvision.models.resnet18(pretrained=False)
    model.fc = nn.Identity()
    return model

In [5]:
def nt_xent_loss(z1, z2, temp=0.5):
    z = torch.cat([z1, z2], dim=0)
    z = nn.functional.normalize(z, dim=1)

    sim = torch.matmul(z, z.T)
    N = z.shape[0]

    mask = torch.eye(N, dtype=torch.bool).to(z.device)
    sim = sim[~mask].view(N, -1)

    positives = torch.sum(z1 * z2, dim=-1)
    positives = torch.cat([positives, positives], dim=0)

    logits = sim / temp
    labels = torch.arange(N).to(z.device)

    loss = nn.CrossEntropyLoss()(logits, labels)
    return loss

In [6]:
def train_ssl(model, dataloader, epochs=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for epoch in range(epochs):
        for x, _ in dataloader:
            x = x.cuda()

            # 2 views
            x1 = x
            x2 = x.flip(-1)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [7]:
def extract_features(model, dataloader):
    model.eval()
    feats, labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.cuda()
            h = model.encoder(x)
            feats.append(h.cpu())
            labels.append(y)

    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

In [8]:
def hierarchical_kmeans_labels(X, k1=50, k2=5):
    # level 1
    km1 = KMeans(n_clusters=k1).fit(X)
    labels1 = km1.labels_

    final_labels = np.zeros(len(X))

    cluster_id = 0

    for i in range(k1):
        idx = np.where(labels1 == i)[0]
        sub_X = X[idx]

        if len(sub_X) < k2:
            final_labels[idx] = cluster_id
            cluster_id += 1
            continue

        km2 = KMeans(n_clusters=k2).fit(sub_X)
        sub_labels = km2.labels_

        for j in range(k2):
            sub_idx = idx[sub_labels == j]
            final_labels[sub_idx] = cluster_id
            cluster_id += 1

    return final_labels.astype(int)     

In [9]:
def resample_by_cluster(dataset, cluster_labels):
    counts = Counter(cluster_labels)
    target = int(np.mean(list(counts.values())))

    indices = []

    for cid, count in counts.items():
        idx = np.where(cluster_labels == cid)[0]

        if count > target:
            selected = np.random.choice(idx, target, replace=False)
        else:
            selected = np.random.choice(idx, target, replace=True)

        indices.extend(selected)

    return Subset(dataset, indices)

In [10]:
# 1. Load data
transform = T.Compose([T.ToTensor()])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# 2. Long-tail
trainset = make_long_tailed(trainset, imbalance_ratio=100)

loader = DataLoader(trainset, batch_size=128, shuffle=True)

# 3. Train SSL warmup
model = SimCLR(get_backbone()).cuda()
train_ssl(model, loader, epochs=50)

# 4. Extract features
features, labels = extract_features(model, loader)

# 5. Clustering
cluster_labels = hierarchical_kmeans_labels(features)

# 6. Resample
resampled_dataset = resample_by_cluster(trainset, cluster_labels)

# 7. Train SSL again
loader_resampled = DataLoader(resampled_dataset, batch_size=128, shuffle=True)

model_resampled = SimCLR(get_backbone()).cuda()
train_ssl(model_resampled, loader_resampled, epochs=100)

100%|██████████| 170M/170M [01:39<00:00, 1.72MB/s] 
c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
def knn_eval(train_feat, train_label, test_feat, test_label):
    knn = KNeighborsClassifier(n_neighbors=20)
    knn.fit(train_feat, train_label)
    return knn.score(test_feat, test_label)

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        return self.fc(x)

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP()
emb2d = reducer.fit_transform(features)

plt.scatter(emb2d[:,0], emb2d[:,1], c=cluster_labels, s=5)
plt.title("Cluster distribution")
plt.show()